In [ ]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorSampler # <-- 1. Swapped to Sampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

# 1. Define the Problem Hamiltonian
hamiltonian = SparsePauliOp.from_list([
    ('II', 42.5),
    ('IZ', 10.5),
    ('ZI', 30.0),
    ('ZZ', -2.0)
])

# 2. Setup the Classical Optimizer and Quantum Sampler
optimizer = COBYLA(maxiter=100)
sampler = StatevectorSampler() # <-- 2. Initialized the Sampler

# 3. Initialize and Run QAOA
# Pass the sampler argument instead of estimator
qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=1) 
print("Running QAOA to find the lowest energy state...")
result = qaoa.compute_minimum_eigenvalue(hamiltonian)

# 4. Interpret the Results
optimal_circuit = result.optimal_circuit.assign_parameters(result.optimal_point)

# Create a copy of the circuit and strip the measurement gates 
# so the Statevector math doesn't crash.
circuit_unmeasured = optimal_circuit.copy()
circuit_unmeasured.remove_final_measurements()

# Calculate state and probabilities on the unmeasured circuit
state = Statevector(circuit_unmeasured)
probabilities = state.probabilities_dict()

# Find the most probable bitstring
best_bitstring = max(probabilities, key=probabilities.get)
best_probability = probabilities[best_bitstring]

# Decode the result
q1 = int(best_bitstring[0])
q0 = int(best_bitstring[1])
x_solution = 2 * q1 + q0

print(f"\n--- Results ---")
print(f"Most probable state: |{best_bitstring}⟩ with probability {best_probability:.2%}")
print(f"Decoded integer value: x = {x_solution}")
print(f"Verifying equation: ({x_solution})^2 - 9 = {x_solution**2 - 9}")

Running QAOA to find the lowest energy state...


c:\Users\Aayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
c:\Users\Aayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\sparse\linalg\_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
c:\Users\Aayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\sparse\_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])



--- Results ---
Most probable state: |11⟩ with probability 28.18%
Decoded integer value: x = 3
Verifying equation: (3)^2 - 9 = 0
